# Decoded-opcode comparator and classical timing

This notebook answers two questions left open by the paper
*Lightweight Machine Learning for Smart-Contract Vulnerability Detection from EVM Bytecode* (v2):

1. **How long does the winning classical model take to fit?** The paper reports the multi-label
   XGBoost result but never recorded its training time. Cell 6 fits it with the paper's exact
   hyperparameters on the full training split and times it.
2. **Does the deep comparator's loss survive a correct input?** The paper found that its
   Conv-Transformer was fed the *hexadecimal text* of each contract, disassembled character by
   character, rather than the decoded bytes. The released rows hold only that legacy text, and it is
   not invertible (the PUSH operands were removed). Cell 3 therefore goes back to the upstream corpus,
   which still carries the deployed bytecode, re-attaches it to every released row by recomputing the
   legacy projection and matching it exactly, and decodes the bytecode properly. Cells 7 and 8 then
   train the paper's best deep configuration (**C2**, `d_model=256`) on the decoded opcodes and, as
   the control, on the legacy tokens, with identical code, seed, rows and internal split.

A third question comes free once the bytecode is back. The paper's models see 67 engineered features,
and nothing had ever tied those columns to the contracts they describe. Cell 4b recomputes them from
the re-attached bytecode and checks each released column against its own extractor output.

**Protocol.** Same released `train_v2` / `val_v2` splits as the paper, verified by SHA-256 against
its manifest. Models are selected and early-stopped on an internal 20% carve-out of `train_v2`;
the reported score is on `val_v2`. **The test split is never opened** (Cell 2 asserts it).
This is therefore a validation-level comparison, not a new test-set result.

**Model code.** Cell 5 is the archive's `02_code/dl_pipeline.py`, embedded verbatim
(SHA-256 `e37b1594fa66d07cd2a3e4335373bbf5a15fff45dc937b337800c787b42d75f1`), so the architecture, losses, optimiser, early stopping and metrics are
the ones behind the paper's Table and Figure 2. Cell 4 is the feature extractor, likewise verbatim.

**Where the paper's numbers come from.** The last cell prints every LaTeX macro this run feeds into
the paper next to the value and the cell that produced it.

**Upstream corpus.** `mwritescode/slither-audited-smart-contracts` on the Hugging Face Hub,
raw files `data/raw/contracts0..8.parquet`, pinned to commit `13594107c7afa216cb0c126f38b8ff6548112dcf`.
Only the `contracts` (address) and `bytecode` columns are read.

**Reference points from the paper's released artifacts** (not recomputed here): legacy C2 on
validation 0.6578 and XGBoost on validation 0.7518, from the released end-to-end run
(`results/full_run_v12/full-ranking.json`); legacy C2 on test 0.6793 (paper, Sec. 6).


In [ ]:
# 1. Configuration and environment
import os, sys, json, time, hashlib, gc, platform, re
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq

SMOKE = os.environ.get("SMOKE", "0") == "1"          # tiny CPU run to check the notebook end to end
RUN_LEGACY_CONTROL = os.environ.get("RUN_LEGACY_CONTROL", "1") == "1"
SEED = 42

KAGGLE_IN = Path("/kaggle/input/defi-bytecode-features-v2")
LOCAL_IN = Path(os.environ.get("V2_DATA_DIR", "../01_data_v2"))
DATA = KAGGLE_IN if KAGGLE_IN.exists() else LOCAL_IN
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUT_DIR / "runs"; RUNS_DIR.mkdir(exist_ok=True)

UPSTREAM_REPO = "mwritescode/slither-audited-smart-contracts"
UPSTREAM_REV = "13594107c7afa216cb0c126f38b8ff6548112dcf"
# The upstream corpus is 1.75 GB; anything under /kaggle/working becomes kernel output, so it
# goes to the session's scratch space instead.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else OUT_DIR
UPSTREAM_DIR = Path(os.environ.get("UPSTREAM_DIR", str(SCRATCH / "upstream")))

FEATURES = ["total_instructions", "unique_instructions", "block_dependent_count", "block_dependency_index", "has_TIMESTAMP", "has_NUMBER", "has_DIFFICULTY", "has_GASLIMIT", "has_COINBASE", "has_BLOCKHASH", "environmental_instructions_count", "environmental_ratio", "unique_environmental_ops", "environmental_complexity", "balance_operations", "address_operations", "caller_operations", "origin_operations", "callvalue_operations", "external_dependency_index", "calldata_size_ops", "calldata_load_ops", "calldata_copy_ops", "total_calldata_ops", "calldata_density", "external_call_count", "has_external_calls", "call_value_ops", "call_gas_limit_ops", "potential_reentrancy_pattern", "pushes", "pops", "stack_imbalance", "stack_operations_ratio", "stack_underflow_risk", "total_gas_cost", "avg_gas_per_instruction", "max_gas_instruction", "high_gas_instructions", "gas_dos_risk_index", "arithmetic_ops_count", "arithmetic_density", "unsafe_arithmetic_pattern", "control_flow_ops", "jumpi_count", "conditional_branching_ratio", "control_flow_complexity", "caller_based_checks", "origin_usage", "access_control_ratio", "uses_origin_instead_caller", "balance_before_external_call", "randomness_ops_count", "has_bad_randomness_pattern", "dangerous_ops_count", "dangerous_ops_density", "opcode_entropy", "reentrancy_risk_score", "frontrunning_risk_score", "dos_risk_score", "arithmetic_risk_score", "overall_security_risk_score", "has_reentrancy_indicators", "has_unchecked_external_calls", "has_arithmetic_vulnerabilities", "has_access_control_issues", "has_dos_vulnerabilities"]
LABELS = ["access-control", "arithmetic", "bad-randomness", "double-spending",
          "locked-ether", "other", "reentrancy", "unchecked-calls"]
EXPECTED_SHA = {"train_v2.parquet": "3a93e58df54e989ec0d18fb480cc47b71a0d9a7ef8d762935594d560f413bcae",
                "val_v2.parquet": "125937bc29ef9658c58dece44a29a576e80f37081f26625fdfd10d1e1f288176"}
EXPECTED_ROWS = {"train_v2.parquet": 89973, "val_v2.parquet": 11247}

try:
    import pyevmasm  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyevmasm==0.2.3"], check=True)
import torch, sklearn, xgboost
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("xgboost", xgboost.__version__, "| sklearn", sklearn.__version__, "| cpus", os.cpu_count())
print("data dir:", DATA, "| SMOKE:", SMOKE, "| legacy control:", RUN_LEGACY_CONTROL)


In [ ]:
# 2. Inputs, bound to the paper's manifest. The test split is never touched.
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def split_path(name):
    assert "test" not in name, "the test split is not part of this protocol"
    return DATA / name

def check_split(name):
    p = split_path(name)
    digest, rows = sha256_file(p), pq.ParquetFile(p).metadata.num_rows
    ok = digest == EXPECTED_SHA[name] and rows == EXPECTED_ROWS[name]
    print(f"{name}: rows {rows:,} | sha256 {digest[:16]}... | matches manifest: {ok}")
    assert ok, "released split does not match the paper's manifest"

def iter_text(name, n_rows=None):
    """Stream the legacy text column row by row; never holds the whole column."""
    seen = 0
    for b in pq.ParquetFile(split_path(name)).iter_batches(batch_size=1024, columns=["bytecode"]):
        for s in b.column("bytecode").to_pylist():
            if n_rows is not None and seen >= n_rows:
                return
            yield s; seen += 1

for name in ("train_v2.parquet", "val_v2.parquet"):
    check_split(name)

# Smoke mode only shrinks the row counts: 2,000/500 rows keep every label positive for the
# classical fit, and the deep runs take the first 240/80 of them.
N_XGB = (2000, 500) if SMOKE else (None, None)
N_DL = (240, 80) if SMOKE else (None, None)
train = pd.read_parquet(split_path("train_v2.parquet"), columns=FEATURES + LABELS)[: N_XGB[0]]
val = pd.read_parquet(split_path("val_v2.parquet"), columns=FEATURES + LABELS)[: N_XGB[1]]
X_tr, y_tr = train[FEATURES].to_numpy("float32"), train[LABELS].to_numpy("float32")
X_va, y_va = val[FEATURES].to_numpy("float32"), val[LABELS].to_numpy("float32")
n_tr_dl, n_va_dl = (N_DL[0] or len(train)), (N_DL[1] or len(val))
print("train", X_tr.shape, "| val", X_va.shape, "| positive rate", round(float((y_tr.max(1) > 0).mean()), 4),
      "| rows for the deep runs:", n_tr_dl, "/", n_va_dl)


In [ ]:
# 3. Recover the deployed bytecode from the upstream corpus, then decode it properly.
#
# What the released text is. The legacy converter disassembled the hexadecimal STRING: each hex
# character became one opcode byte (ASCII '0'..'9' -> 0x30..0x39 = ADDRESS..CODECOPY, 'a'..'f' ->
# 0x61..0x66 = PUSH2..PUSH7) and each PUSHn swallowed the following n characters as its operand; the
# operands were then removed, and a trailing PUSH whose operand ran past the end of the string was
# dropped. That projection loses the swallowed characters, so the text cannot be inverted -- but it
# can be recomputed from the original hex string, which gives an exact join key.
NAME = {"0": "ADDRESS", "1": "BALANCE", "2": "ORIGIN", "3": "CALLER", "4": "CALLVALUE",
        "5": "CALLDATALOAD", "6": "CALLDATASIZE", "7": "CALLDATACOPY", "8": "CODESIZE",
        "9": "CODECOPY", "a": "PUSH2", "b": "PUSH3", "c": "PUSH4", "d": "PUSH5", "e": "PUSH6", "f": "PUSH7"}
SKIP = {"a": 2, "b": 3, "c": 4, "d": 5, "e": 6, "f": 7}

PREFIX_TOKENS = 120
# Hex characters behind one projected token: a digit stands for itself, PUSHn also swallowed n.
TOKEN_CHARS = {NAME[c]: 1 + SKIP.get(c, 0) for c in NAME}
MAX_DROPPED = 7               # a trailing PUSH the converter could not complete

def legacy_projection(h, limit=None):
    out, i, n = [], 0, len(h)
    while i < n and (limit is None or len(out) < limit):
        c = h[i]; w = SKIP.get(c, 0)
        if i + w >= n: break        # a trailing PUSH without its full operand was dropped by the converter
        out.append(NAME[c]); i += 1 + w
    return " ".join(out)

def clean_tokens(text):       # released text: drop the rare surviving operand token (only ever the last one)
    return text.split() if "0x" not in text else [t for t in text.split() if not t.startswith("0x")]

def sha1(s):
    return hashlib.sha1(s.encode()).hexdigest()

t0 = time.time()
keys, prefixes = {}, {}
for split, name, n_rows in (("train", "train_v2.parquet", n_tr_dl), ("val", "val_v2.parquet", n_va_dl)):
    keys[split] = []
    for s in iter_text(name, n_rows):
        toks = clean_tokens(s)
        key = sha1(" ".join(toks))
        keys[split].append(key)
        implied = sum(TOKEN_CHARS[t] for t in toks)     # hex characters this row accounts for
        prefixes.setdefault(sha1(" ".join(toks[:PREFIX_TOKENS])), {})[key] = implied
wanted = set(keys["train"]) | set(keys["val"])
print(f"released rows keyed: train {len(keys['train']):,} | val {len(keys['val']):,} | "
      f"distinct keys {len(wanted):,} | distinct {PREFIX_TOKENS}-token prefixes {len(prefixes):,} | {time.time()-t0:.0f}s")

def upstream_file(i):
    for p in (UPSTREAM_DIR / f"contracts{i}.parquet", UPSTREAM_DIR / "data" / "raw" / f"contracts{i}.parquet"):
        if p.exists(): return p
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(UPSTREAM_REPO, f"data/raw/contracts{i}.parquet", repo_type="dataset",
                                revision=UPSTREAM_REV, local_dir=str(UPSTREAM_DIR)))

# Each upstream contract is projected to PREFIX_TOKENS tokens, which is cheap; the full
# projection is paid for only when both that prefix and the implied hex length match a released
# row. The prefix alone rejects little -- every solc contract opens the same way -- so the length
# is what makes the scan cheap.
t0 = time.time()
code_by_key, n_up, n_unusable, n_collide, n_candidates = {}, 0, 0, 0, 0
for i in range(9):
    pf = pq.ParquetFile(upstream_file(i))
    for b in pf.iter_batches(batch_size=2048, columns=["contracts", "bytecode"]):
        for addr, h in zip(b.column("contracts").to_pylist(), b.column("bytecode").to_pylist()):
            n_up += 1
            h = (h[2:] if h and h.startswith("0x") else h or "").lower()
            if not h or len(h) % 2 or not re.fullmatch(r"[0-9a-f]+", h):
                n_unusable += 1; continue
            hits = prefixes.get(sha1(legacy_projection(h, PREFIX_TOKENS)))
            if not hits: continue
            # length is exact and free: reject before paying for the full projection
            hits = {k for k, implied in hits.items() if 0 <= len(h) - implied <= MAX_DROPPED}
            if not hits: continue
            n_candidates += 1
            k = sha1(legacy_projection(h))
            if k in hits:
                code = bytes.fromhex(h)
                if k in code_by_key: n_collide += code_by_key[k][1] != code
                else: code_by_key[k] = (addr, code)
    print(f"  contracts{i}: upstream rows {n_up:,} | re-attached keys {len(code_by_key):,}/{len(wanted):,}")
recovery = {"upstream_rows": n_up, "upstream_unusable": n_unusable, "released_keys": len(wanted),
            "keys_reattached": len(code_by_key), "same_projection_different_bytecode": n_collide,
            "prefix_candidates": n_candidates, "upstream_revision": UPSTREAM_REV}
for split in ("train", "val"):
    hit = sum(k in code_by_key for k in keys[split])
    recovery[f"{split}_rows"] = len(keys[split]); recovery[f"{split}_rows_reattached"] = hit
    print(f"{split}: {hit:,}/{len(keys[split]):,} rows re-attached ({100*hit/len(keys[split]):.2f}%)")
print(f"upstream contracts {n_up:,} | unusable {n_unusable:,} | projection collisions with different bytecode: {n_collide} | {time.time()-t0:.0f}s")
# `code_by_key` holds the re-attached bytecode of every matched row; cell 7 frees it as soon as
# the decoded arm is encoded.

# Proper decoding: bytes -> one mnemonic per instruction (pyevmasm's Istanbul table), PUSH operands
# skipped (constants and addresses would make the vocabulary open-ended), unknown bytes -> INVALID.
# The Solidity metadata trailer (CBOR, length in the last two bytes) is not code and is stripped when
# its marker is where the declared length says it is.
from pyevmasm import instruction_tables
_T = instruction_tables["istanbul"]; OPNAME = {}
for op in range(256):
    try: OPNAME[op] = _T[op].name
    except KeyError: pass

def strip_metadata(code):
    if len(code) >= 4:
        n = int.from_bytes(code[-2:], "big")
        if 8 <= n <= 120 and n + 2 < len(code):
            trailer = code[-(n + 2):]
            if b"ipfs" in trailer or b"bzzr" in trailer:
                return code[: -(n + 2)], True
    return code, False

def decode_opcodes(code):
    out, i, n = [], 0, len(code)
    while i < n:
        op = code[i]; out.append(OPNAME.get(op, "INVALID")); i += 1 + (op - 0x5f if 0x60 <= op <= 0x7f else 0)
    return " ".join(out)

def decoded_text(k):
    code, stripped = strip_metadata(code_by_key[k][1])
    return decode_opcodes(code), stripped

# sanity on the re-attached bytecode: the solc preamble PUSH1 0x80 PUSH1 0x40 MSTORE, and no
# INVALID/STOP runs that a mis-aligned decode would produce
sample = [k for k in keys["train"][:2000] if k in code_by_key]
dec = [decoded_text(k) for k in sample]
pre = sum(code_by_key[k][1][:5] in (bytes.fromhex("6080604052"), bytes.fromhex("6060604052")) for k in sample)
lens = np.array([d.count(" ") + 1 for d, _ in dec]); inv = np.array([d.split().count("INVALID") for d, _ in dec])
recovery.update({"sample_rows": len(sample), "sample_solc_preamble": int(pre), "sample_metadata_stripped": int(sum(s for _, s in dec)),
                 "sample_decoded_len_median": int(np.median(lens)), "sample_invalid_per_contract_median": float(np.median(inv))})
print(f"sample of {len(sample):,} train rows: solc preamble {pre:,} | metadata stripped {sum(s for _, s in dec):,} | "
      f"decoded tokens median {int(np.median(lens)):,} (max {lens.max():,}) | INVALID per contract median {np.median(inv):.0f}")
print("example:", dec[0][0][:160])
del sample, dec; gc.collect()


## 4. Do the released features describe these contracts?

The paper's models see all 67 engineered features, not the sequences, and until now nothing
tied those columns to the actual deployed bytecode: the paper says so in its limitations. The
re-attached bytecode makes the check possible. The released columns are standardised, so they cannot
be compared value by value; standardisation is affine and increasing, so if a released column is the
extractor's output for the same contract then a least-squares fit of the released column on the
recomputed one has $R^2 = 1$ and a positive slope. Anything less means the column is not that
function of this bytecode.

The next cell is the feature extractor, embedded verbatim
(SHA-256 `6ba62190e829d064265267159fcaf68deb123ccf9a526719b9240ab3fdc2b87b`), followed by the check on a sample of training contracts.

In [ ]:
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, TransformerMixin
from pyevmasm import disassemble_all
import multiprocessing as mp
import numpy as np
from scipy.stats import entropy
from collections import Counter
import pandas as pd

class EVMBytecodeFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Feature transformer from EVM bytecode for smart-contract vulnerability detection.
    """
    def __init__(self, bytecode_column="bytecode", n_workers=None):
        self.bytecode_column = bytecode_column
        self.n_workers = n_workers
        # Fixed list of all output features
        self.feature_names_ = [
            # Basic
            "total_instructions", "unique_instructions",
            # Block dependence
            "block_dependent_count", "block_dependency_index",
            "has_TIMESTAMP", "has_NUMBER", "has_DIFFICULTY", "has_GASLIMIT",
            "has_COINBASE", "has_BLOCKHASH",
            # Environmental
            "environmental_instructions_count", "environmental_ratio",
            "unique_environmental_ops", "environmental_complexity",
            # Specific ops
            "balance_operations", "address_operations", "caller_operations",
            "origin_operations", "callvalue_operations", "external_dependency_index",
            # Calldata
            "calldata_size_ops", "calldata_load_ops", "calldata_copy_ops",
            "total_calldata_ops", "calldata_density",
            # External calls
            "external_call_count", "has_external_calls", "call_value_ops",
            "call_gas_limit_ops", "potential_reentrancy_pattern",
            # Stack
            "pushes", "pops", "stack_imbalance", "stack_operations_ratio",
            "stack_underflow_risk",
            # Gas
            "total_gas_cost", "avg_gas_per_instruction", "max_gas_instruction",
            "high_gas_instructions", "gas_dos_risk_index",
            # Arithmetic
            "arithmetic_ops_count", "arithmetic_density", "unsafe_arithmetic_pattern",
            # Control flow
            "control_flow_ops", "jumpi_count", "conditional_branching_ratio",
            "control_flow_complexity",
            # Access control
            "caller_based_checks", "origin_usage", "access_control_ratio",
            "uses_origin_instead_caller",
            # Advanced patterns
            "balance_before_external_call", "randomness_ops_count",
            "has_bad_randomness_pattern",
            # Complexity
            "dangerous_ops_count", "dangerous_ops_density", "opcode_entropy",
            # Composite scores
            "reentrancy_risk_score", "frontrunning_risk_score",
            "dos_risk_score", "arithmetic_risk_score", "overall_security_risk_score",
            # Binary flags
            "has_reentrancy_indicators", "has_unchecked_external_calls",
            "has_arithmetic_vulnerabilities", "has_access_control_issues",
            "has_dos_vulnerabilities",
        ]

    def _extract_features_single(self, bytecode) -> dict:
        """Extract features from a single bytecode (hex string or bytes)"""
        if isinstance(bytecode, str):
            bytecode = bytecode.strip()
            if bytecode.startswith("0x"):
                bytecode = bytecode[2:]
            if not bytecode:
                bytecode_bytes = b""
            else:
                try:
                    bytecode_bytes = bytes.fromhex(bytecode)
                except ValueError:
                    bytecode_bytes = b""
        else:
            bytecode_bytes = bytes(bytecode) if bytecode is not None else b""

        try:
            instructions = list(disassemble_all(bytecode_bytes))
        except Exception:
            instructions = []
        n = len(instructions)
        if n == 0:
            return {name: 0.0 for name in self.feature_names_}

        # Mnemonic frequency counter
        mnemonic_counter = Counter(instr.mnemonic for instr in instructions)

        # Lookup sets for fast membership checks
        block_dependent = {"TIMESTAMP", "NUMBER", "DIFFICULTY", "GASLIMIT", "COINBASE", "BLOCKHASH"}
        call_ops = {"CALL", "DELEGATECALL", "STATICCALL", "CALLCODE"}
        arithmetic_ops = {"ADD", "SUB", "MUL", "DIV", "MOD", "SDIV", "SMOD", "EXP", "SIGNEXTEND"}
        dangerous_ops = block_dependent.union(call_ops).union(arithmetic_ops).union({"SELFDESTRUCT"})
        randomness_ops = {"BLOCKHASH", "TIMESTAMP", "DIFFICULTY", "COINBASE"}

        # Basic aggregates
        total_instructions = n
        unique_instructions = len(mnemonic_counter)

        block_dependent_count = sum(mnemonic_counter.get(op, 0) for op in block_dependent)

        environmental_count = sum(1 for instr in instructions if getattr(instr, "group", None) == "Environmental Information")
        unique_env_ops = len({instr.mnemonic for instr in instructions if getattr(instr, "group", None) == "Environmental Information"})

        balance_ops = mnemonic_counter.get("BALANCE", 0)
        caller_ops = mnemonic_counter.get("CALLER", 0)
        origin_ops = mnemonic_counter.get("ORIGIN", 0)
        callvalue_ops = mnemonic_counter.get("CALLVALUE", 0)

        calldata_size = mnemonic_counter.get("CALLDATASIZE", 0)
        calldata_load = mnemonic_counter.get("CALLDATALOAD", 0)
        calldata_copy = mnemonic_counter.get("CALLDATACOPY", 0)

        external_call_count = sum(mnemonic_counter.get(op, 0) for op in call_ops)
        gas_ops = mnemonic_counter.get("GAS", 0)  # often precedes CALL

        # Aggregates over instruction attributes
        pushes = sum(getattr(instr, "pushes", 0) for instr in instructions)
        pops = sum(getattr(instr, "pops", 0) for instr in instructions)
        fees = [getattr(instr, "fee", 0) for instr in instructions]

        total_gas = sum(fees)
        avg_gas = total_gas / n if n else 0
        max_gas = max(fees) if fees else 0
        high_gas_count = sum(f > 1000 for f in fees)

        # Simple PC-based patterns (heuristic; imperfect under JUMP, but informative)
        arithmetic_ops_set = {"ADD", "SUB", "MUL", "DIV", "MOD", "SDIV", "SMOD", "EXP", "SIGNEXTEND"}
        target_mnemonics = {"SSTORE", "CALL", "DELEGATECALL", "STATICCALL", "CALLCODE", "BALANCE", "JUMPI"} | arithmetic_ops_set

        # Keep only those present in the contract
        target_mnemonics = target_mnemonics & mnemonic_counter.keys()

        pcs = {
            mnemonic: sorted(instr.pc for instr in instructions if instr.mnemonic == mnemonic)
            for mnemonic in target_mnemonics
        }

        potential_reentrancy = 0
        if "SSTORE" in pcs and any(op in pcs for op in call_ops):
            for s_pc in pcs["SSTORE"]:
                for c_op in call_ops:
                    for c_pc in pcs.get(c_op, []):
                        if c_pc > s_pc and (c_pc - s_pc) < 20:
                            potential_reentrancy = 1
                            break

        unsafe_arith = 0
        jumpi_pcs = pcs.get("JUMPI", [])
        for op in arithmetic_ops:
            for a_pc in pcs.get(op, []):
                for j_pc in jumpi_pcs:
                    if j_pc > a_pc and (j_pc - a_pc) < 5:
                        unsafe_arith = 1
                        break

        balance_before_call = 0
        balance_pcs = pcs.get("BALANCE", [])
        if balance_pcs:
            for b_pc in balance_pcs:
                for c_op in call_ops:
                    for c_pc in pcs.get(c_op, []):
                        if c_pc > b_pc and (c_pc - b_pc) < 10:
                            balance_before_call = 1
                            break

        # Assemble the feature dictionary
        features = {
            "total_instructions": total_instructions,
            "unique_instructions": unique_instructions,
            "block_dependent_count": block_dependent_count,
            "block_dependency_index": block_dependent_count / total_instructions,
            **{f"has_{op}": int(op in mnemonic_counter) for op in block_dependent},
            "environmental_instructions_count": environmental_count,
            "environmental_ratio": environmental_count / total_instructions,
            "unique_environmental_ops": unique_env_ops,
            "environmental_complexity": unique_env_ops * (environmental_count / total_instructions),
            "balance_operations": balance_ops,
            "address_operations": mnemonic_counter.get("ADDRESS", 0),
            "caller_operations": caller_ops,
            "origin_operations": origin_ops,
            "callvalue_operations": callvalue_ops,
            "external_dependency_index": (block_dependent_count + balance_ops) / total_instructions,
            "calldata_size_ops": calldata_size,
            "calldata_load_ops": calldata_load,
            "calldata_copy_ops": calldata_copy,
            "total_calldata_ops": calldata_size + calldata_load + calldata_copy,
            "calldata_density": (calldata_size + calldata_load + calldata_copy) / total_instructions,
            "external_call_count": external_call_count,
            "has_external_calls": int(external_call_count > 0),
            "call_value_ops": callvalue_ops,
            "call_gas_limit_ops": gas_ops,
            "potential_reentrancy_pattern": potential_reentrancy,
            "pushes": pushes,
            "pops": pops,
            "stack_imbalance": pushes - pops,
            "stack_operations_ratio": pops / max(1, pushes),
            "stack_underflow_risk": int(pushes - pops < 0),
            "total_gas_cost": total_gas,
            "avg_gas_per_instruction": avg_gas,
            "max_gas_instruction": max_gas,
            "high_gas_instructions": high_gas_count,
            "gas_dos_risk_index": high_gas_count / total_instructions,
            "arithmetic_ops_count": sum(mnemonic_counter.get(op, 0) for op in arithmetic_ops),
            "arithmetic_density": sum(mnemonic_counter.get(op, 0) for op in arithmetic_ops) / total_instructions,
            "unsafe_arithmetic_pattern": unsafe_arith,
            "control_flow_ops": sum(mnemonic_counter.get(op, 0) for op in {"JUMP", "JUMPI", "RETURN", "REVERT", "STOP", "INVALID"}),
            "jumpi_count": mnemonic_counter.get("JUMPI", 0),
            "conditional_branching_ratio": mnemonic_counter.get("JUMPI", 0) / max(1, sum(mnemonic_counter.get(op, 0) for op in {"JUMP", "JUMPI"})),
            "control_flow_complexity": mnemonic_counter.get("JUMPI", 0) ** 2 / total_instructions,
            "caller_based_checks": caller_ops,
            "origin_usage": origin_ops,
            "access_control_ratio": caller_ops / max(1, external_call_count),
            "uses_origin_instead_caller": int(origin_ops > caller_ops),
            "balance_before_external_call": balance_before_call,
            "randomness_ops_count": sum(mnemonic_counter.get(op, 0) for op in randomness_ops),
            "has_bad_randomness_pattern": int(sum(mnemonic_counter.get(op, 0) for op in randomness_ops) > 0),
            "dangerous_ops_count": sum(mnemonic_counter.get(op, 0) for op in dangerous_ops),
            "dangerous_ops_density": sum(mnemonic_counter.get(op, 0) for op in dangerous_ops) / total_instructions,
            "opcode_entropy": entropy(list(mnemonic_counter.values())) if len(mnemonic_counter) > 1 else 0.0,
        }

        # Composite risk scores
        reentrancy_score = (
            features["external_call_count"] +
            features["call_value_ops"] +
            features["potential_reentrancy_pattern"] +
            features["balance_before_external_call"]
        ) / total_instructions

        frontrunning_score = (
            features["block_dependent_count"] +
            features["external_dependency_index"] +
            features["has_bad_randomness_pattern"]
        ) / total_instructions

        dos_score = (
            features["gas_dos_risk_index"] +
            features["high_gas_instructions"] +
            features["control_flow_complexity"] +
            features["stack_underflow_risk"]
        ) / total_instructions

        arith_score = (
            features["arithmetic_ops_count"] +
            features["unsafe_arithmetic_pattern"] +
            features["stack_underflow_risk"]
        ) / total_instructions

        overall_score = np.mean([
            reentrancy_score, frontrunning_score, dos_score, arith_score,
            features["dangerous_ops_density"], features["external_dependency_index"]
        ])

        features.update({
            "reentrancy_risk_score": reentrancy_score,
            "frontrunning_risk_score": frontrunning_score,
            "dos_risk_score": dos_score,
            "arithmetic_risk_score": arith_score,
            "overall_security_risk_score": overall_score,
            "has_reentrancy_indicators": int(reentrancy_score > 0.1),
            "has_unchecked_external_calls": int(external_call_count > features["jumpi_count"]),
            "has_arithmetic_vulnerabilities": int(features["unsafe_arithmetic_pattern"] > 0),
            "has_access_control_issues": int(features["access_control_ratio"] < 0.2 and external_call_count > 0),
            "has_dos_vulnerabilities": int(features["gas_dos_risk_index"] > 0.1),
        })

        # Guarantee the full, ordered feature set
        return {name: features.get(name, 0.0) for name in self.feature_names_}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            bytecodes = X[self.bytecode_column].values
            index = X.index
        else:
            bytecodes = np.asarray(X)
            index = None

        n_jobs = self.n_workers or max(1, mp.cpu_count() - 1)

        results = Parallel(n_jobs=n_jobs)(
            delayed(self._extract_features_single)(bc) for bc in bytecodes
        )

        return pd.DataFrame(results, columns=self.feature_names_, index=index)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


In [ ]:
# 4b. The check itself.
N_PROV = 40 if SMOKE else int(os.environ.get("N_PROVENANCE", "1200"))
prov_keys = [k for k in keys["train"] if k in code_by_key][:N_PROV]
prov_rows = [i for i, k in enumerate(keys["train"]) if k in code_by_key][:N_PROV]
prov_hex = [code_by_key[k][1].hex() for k in prov_keys]

t0 = time.time()
extractor = EVMBytecodeFeatureExtractor(bytecode_column="bytecode", n_workers=os.cpu_count())
mine = extractor.transform(pd.DataFrame({"bytecode": prov_hex}))
if not isinstance(mine, pd.DataFrame):
    mine = pd.DataFrame(mine, columns=extractor.feature_names_)
rel = train[FEATURES].iloc[prov_rows].reset_index(drop=True)
print(f"recomputed {mine.shape[0]:,} contracts x {mine.shape[1]} features in {time.time()-t0:.0f}s")

exact, imperfect, flat = [], [], []
for f in FEATURES:
    x, y = mine[f].to_numpy("float64"), rel[f].to_numpy("float64")
    if np.std(x) == 0 or np.std(y) == 0:
        flat.append(f); continue
    a, b = np.polyfit(x, y, 1)
    r2 = 1 - float(np.sum((y - (a * x + b)) ** 2) / np.sum((y - y.mean()) ** 2))
    (exact if r2 > 1 - 1e-9 and a > 0 else imperfect).append((f, r2, float(a)))
print(f"released = affine(recomputed) exactly: {len(exact)}/{len(FEATURES)} features")
print(f"constant across the sample (no test possible): {len(flat)} {flat}")
for f, r2, a in sorted(imperfect, key=lambda t: t[1])[:10]:
    print(f"   NOT an affine image: {f:34s} R2 {r2:.6f} slope {a:.4g}")
provenance = {"rows": len(prov_hex), "features_exact": len(exact), "features_total": len(FEATURES),
              "features_constant_in_sample": flat,
              "features_not_affine": [{"feature": f, "r2": r2, "slope": a} for f, r2, a in imperfect]}
(OUT_DIR / "feature_provenance.json").write_text(json.dumps(provenance, indent=2))
del mine, rel, prov_hex; gc.collect()


## 5. Model code, verbatim from the archive

The next cell is `02_code/dl_pipeline.py` from the paper's archive, unchanged
(SHA-256 `e37b1594fa66d07cd2a3e4335373bbf5a15fff45dc937b337800c787b42d75f1`). It defines the tokeniser, the Conv-Transformer,
the losses, the training loop with early stopping, the metrics, and the configuration registry
`DL_EXPERIMENTS` from which C2 is taken.

In [ ]:
"""Conv-Transformer multi-label pipeline for the reproducibility companion
notebook.

Faithful port of the proven code from the original ML_Experiments.ipynb,
restructured as a clean importable module so that the Kaggle paper-
companion notebook stays narrative-readable.

Public entrypoints
------------------
- :data:`DL_EXPERIMENTS`     — registry of 10 ablation configurations
                                 (A1 baseline + 5 loss variants + 4 archs)
- :func:`run_dl_experiment`  — train one config, cache to ``runs/{name}.json``
- :func:`run_all_dl`         — loop ``DL_EXPERIMENTS`` with defensive
                                 try/except per run + W&B subnamespace logs
- :func:`build_token_ids`    — one-off tokeniser + sequence cache

Design notes
------------
* PyTorch inference mode is invoked via ``model.train(False)`` which is
  functionally equivalent to the conventional inference-mode toggle but
  avoids false-positive matches by code scanners that look for that name.
* Every public function is idempotent given the seed and the existing
  ``runs/`` cache.  Defensive: one experiment failing does not abort the
  others.
"""
from __future__ import annotations

import json
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SEQ_LEN_FULL = 20000


# ============================================================================
# 1. Tokeniser
# ============================================================================
class BytecodeTokenizer:
    """Whitespace-separated bytecode opcode tokeniser.  PAD=0, UNK=1."""
    PAD, UNK = 0, 1

    def __init__(self, sep: str = " ") -> None:
        self.sep = sep
        self.token2id: Dict[str, int] = {"<PAD>": self.PAD, "<UNK>": self.UNK}

    def fit(self, texts: Sequence[str]) -> "BytecodeTokenizer":
        cnt: Counter = Counter()
        for t in texts:
            cnt.update(str(t).split(self.sep))
        for tok in cnt.keys():
            if tok not in self.token2id:
                self.token2id[tok] = len(self.token2id)
        return self

    def encode(self, text: str, max_len: int) -> List[int]:
        ids = [self.token2id.get(t, self.UNK) for t in str(text).split(self.sep)]
        ids = ids[:max_len]
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def encode_unpadded(self, text: str, max_len: int) -> np.ndarray:
        """Truncate but do not pad — used for ragged storage."""
        ids = [self.token2id.get(t, self.UNK) for t in str(text).split(self.sep)]
        if len(ids) > max_len:
            ids = ids[:max_len]
        return np.asarray(ids, dtype=np.int32)

    @property
    def vocab_size(self) -> int:
        return len(self.token2id)


def build_token_ids(series: pd.Series, tokenizer: BytecodeTokenizer,
                    max_len: int = MAX_SEQ_LEN_FULL,
                    progress_every: int = 10_000) -> List[np.ndarray]:
    """Encode a Series of bytecode strings into a ragged List[np.ndarray].

    Each element is a 1-D int32 array, truncated at ``max_len`` but NOT
    padded. Padding is deferred to :class:`BytecodeDataset.__getitem__`
    so per-experiment ``max_seq_len`` does not pay the worst-case bill,
    and overall RAM scales with actual token counts (~10x cheaper than
    a fixed (N, MAX_SEQ_LEN_FULL) array on real Ethereum bytecode).
    """
    out: List[np.ndarray] = [None] * len(series)  # type: ignore[list-item]
    for i, txt in enumerate(series.values):
        out[i] = tokenizer.encode_unpadded(txt, max_len)
        if (i + 1) % progress_every == 0:
            print(f"    encoded {i+1}/{len(series)}", flush=True)
    return out


# ============================================================================
# 2. Architecture
# ============================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int, stride: int) -> None:
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, k, stride=stride, padding=k // 2)
        self.norm = nn.BatchNorm1d(out_ch)
        self.proj = (nn.Conv1d(in_ch, out_ch, 1, stride=stride)
                     if (in_ch != out_ch or stride > 1) else None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.proj(x) if self.proj is not None else x
        return F.gelu(self.norm(self.conv(x)) + res)


class AttentionPooling(nn.Module):
    """Scaled-dot-product attention pool with padding mask support."""
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.q = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor,
                mask: torch.Tensor | None = None) -> torch.Tensor:
        if mask is not None:
            valid = (~mask).float().unsqueeze(-1)
            x_sum = (x * valid).sum(1, keepdim=True)
            denom = valid.sum(1, keepdim=True).clamp(min=1)
            mean = x_sum / denom
        else:
            mean = x.mean(1, keepdim=True)
        q = self.q(mean)
        scale = x.size(-1) ** 0.5
        scores = (q @ x.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1), -1e9)
        return (scores.softmax(-1) @ x).squeeze(1)


class BytecodeEncoder(nn.Module):
    """Three modes via cfg: Conv-Transformer (default), pure CNN, pure Transformer."""
    def __init__(self, vocab_size: int, cfg: Dict[str, Any]) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg["d_model"]
        self.embed = nn.Embedding(vocab_size, d, padding_idx=0)
        self.pure_cnn = bool(cfg.get("pure_cnn", False))
        self.pure_tf = bool(cfg.get("pure_transformer", False))
        if not self.pure_tf:
            self.cnn = nn.Sequential(
                ConvBlock(d, d, 7, 4),
                ConvBlock(d, d, 7, 5),
                ConvBlock(d, d, 5, 4),
                ConvBlock(d, d, 5, 2),
            )
            self.short_len = 125
        else:
            self.cnn = None
            self.short_len = cfg["max_seq_len"]
        if not self.pure_cnn:
            enc = nn.TransformerEncoderLayer(
                d_model=d, nhead=8, dim_feedforward=512,
                dropout=cfg["dropout"], batch_first=True, norm_first=True)
            self.transformer = nn.TransformerEncoder(
                enc, num_layers=cfg["n_transformer_layers"])
        else:
            self.transformer = None
        self.pool = AttentionPooling(d)

    def _downsample_mask(self, pad_mask: torch.Tensor, target_len: int) -> torch.Tensor:
        pad_f = pad_mask.float().unsqueeze(1)
        pooled = F.adaptive_avg_pool1d(pad_f, target_len).squeeze(1)
        return pooled > 0.99

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad_full = (x == 0)
        x = self.embed(x)
        if self.cnn is not None:
            x = self.cnn(x.transpose(1, 2)).transpose(1, 2)
            pad = self._downsample_mask(pad_full, self.short_len)
        else:
            pad = pad_full
        all_pad = pad.all(dim=1)
        if all_pad.any():
            pad[all_pad, 0] = False
        if self.transformer is not None:
            x = self.transformer(x, src_key_padding_mask=pad)
        return self.pool(x, mask=pad)


class MultilabelClassifier(nn.Module):
    def __init__(self, vocab_size: int, n_num: int, n_labels: int,
                 cfg: Dict[str, Any]) -> None:
        super().__init__()
        self.use_seq = bool(cfg["use_sequence"])
        self.use_num = bool(cfg["use_num_features"]) and n_num > 0
        d = cfg["d_model"]
        if self.use_seq:
            self.encoder = BytecodeEncoder(vocab_size, cfg)
            seq_d = d
        else:
            self.encoder = None
            seq_d = 0
        if self.use_num:
            self.num_mlp = nn.Sequential(
                nn.Linear(n_num, d), nn.LayerNorm(d), nn.GELU(),
                nn.Linear(d, d))
            num_d = d
        else:
            self.num_mlp = None
            num_d = 0
        self.drop = nn.Dropout(cfg["dropout"])
        self.head = nn.Linear(seq_d + num_d, n_labels)

    def forward(self, x_seq: torch.Tensor,
                x_num: torch.Tensor) -> torch.Tensor:
        embs = []
        if self.use_seq:
            embs.append(self.encoder(x_seq))
        if self.use_num and x_num.size(1) > 0:
            embs.append(self.num_mlp(x_num))
        emb = torch.cat(embs, dim=1) if len(embs) > 1 else embs[0]
        return self.head(self.drop(emb))


# ============================================================================
# 3. Loss functions and metrics
# ============================================================================
def compute_pos_weight(labels: torch.Tensor, device: str) -> torch.Tensor:
    n = len(labels)
    pos = labels.sum(0).clamp(min=1)
    return ((n - pos) / pos).to(device)


def get_loss_fn(cfg: Dict[str, Any], pos_weight: torch.Tensor | None = None):
    name = cfg["loss"]
    use_pw = bool(cfg["use_pos_weight"])
    if name == "bce":
        pw = pos_weight if use_pw else None
        return lambda lg, y: F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
    if name == "focal":
        gamma = float(cfg["focal_gamma"])
        pw = pos_weight if use_pw else None
        def fl(lg, y):
            bce = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw,
                                                     reduction="none")
            p = torch.sigmoid(lg)
            p_t = p * y + (1 - p) * (1 - y)
            return ((1 - p_t) ** gamma * bce).mean()
        return fl
    if name == "asymmetric":
        gp, gn = float(cfg["asym_gamma_pos"]), float(cfg["asym_gamma_neg"])
        clip = float(cfg["asym_clip"])
        def al(lg, y):
            p = torch.sigmoid(lg)
            p_neg = (p - clip).clamp(min=0) if clip > 0 else p
            log_p = torch.log(p.clamp(min=1e-8))
            log_1mp = torch.log((1 - p_neg).clamp(min=1e-8))
            return (-y * log_p * (1 - p) ** gp
                    - (1 - y) * log_1mp * p_neg ** gn).mean()
        return al
    raise ValueError(f"unknown loss: {name}")


def compute_dl_metrics(logits: torch.Tensor, labels: torch.Tensor,
                       thresholds=None) -> Dict[str, Any]:
    probs = torch.sigmoid(logits)
    if thresholds is None:
        thresholds = 0.5
    if isinstance(thresholds, (list, np.ndarray)):
        thresholds = torch.tensor(thresholds, dtype=probs.dtype, device=probs.device)
    preds = (probs > thresholds).float()
    tp = (preds * labels).sum(0)
    fp = (preds * (1 - labels)).sum(0)
    fn = ((1 - preds) * labels).sum(0)
    prec = tp / (tp + fp).clamp(min=1e-8)
    rec = tp / (tp + fn).clamp(min=1e-8)
    f1 = 2 * prec * rec / (prec + rec).clamp(min=1e-8)
    return {
        "f1_macro": f1.mean().item(),
        "f1_per_label": f1.tolist(),
        "subset_acc": (preds == labels).all(dim=1).float().mean().item(),
        "precision_per_label": prec.tolist(),
        "recall_per_label": rec.tolist(),
    }


def tune_thresholds(probs: torch.Tensor, labels: torch.Tensor,
                    n_steps: int = 33) -> np.ndarray:
    grid = np.linspace(0.05, 0.95, n_steps)
    best = np.full(labels.shape[1], 0.5)
    for c in range(labels.shape[1]):
        best_f1 = 0.0
        for t in grid:
            pr = (probs[:, c] > t).float()
            tp = (pr * labels[:, c]).sum()
            fp = (pr * (1 - labels[:, c])).sum()
            fn = ((1 - pr) * labels[:, c]).sum()
            p = tp / (tp + fp).clamp(min=1e-8)
            r = tp / (tp + fn).clamp(min=1e-8)
            f = (2 * p * r / (p + r).clamp(min=1e-8)).item()
            if f > best_f1:
                best_f1, best[c] = f, float(t)
    return best


# ============================================================================
# 4. Dataset + DataLoader
# ============================================================================
class BytecodeDataset(Dataset):
    """Multi-label dataset over ragged token ids.

    ``token_ids_arr`` may be either:
      * a 2-D ``np.ndarray`` of shape ``(N, MAX_SEQ_LEN_FULL)`` already
        zero-padded (legacy fixed-pad layout), OR
      * a ``List[np.ndarray]`` of variable-length int32 sequences
        (ragged layout — preferred, ~10x cheaper RAM on real bytecode).

    ``__getitem__`` slices to ``max_len`` and pads with zeros so every
    sample in a batch has the same shape (``collate`` uses ``torch.stack``).
    """
    def __init__(self, labels: np.ndarray, nums: np.ndarray | None,
                 max_len: int, token_ids_arr,
                 token_aug_prob: float = 0.0) -> None:
        self.labels = labels.astype(np.float32)
        self.nums = nums.astype(np.float32) if nums is not None else None
        self.max_len = max_len
        self.tok_ids = token_ids_arr
        self.aug_p = token_aug_prob

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int):
        raw = self.tok_ids[i]
        # `raw` is a 1-D ndarray either way (list[ndarray] or 2D ndarray row).
        ids = raw[:self.max_len]
        if ids.dtype != np.int32:
            ids = ids.astype(np.int32, copy=False)
        if len(ids) < self.max_len:
            pad = np.zeros(self.max_len - len(ids), dtype=np.int32)
            ids = np.concatenate([ids, pad])
        if self.aug_p > 0:
            ids = ids.copy()
            m = (np.random.random(len(ids)) < self.aug_p) & (ids != 0)
            ids[m] = 1
        x_seq = torch.tensor(ids, dtype=torch.long)
        x_num = (torch.tensor(self.nums[i], dtype=torch.float)
                 if self.nums is not None else torch.empty(0))
        y = torch.tensor(self.labels[i], dtype=torch.float)
        return x_seq, x_num, y


def collate(batch):
    xs, xn, y = zip(*batch)
    return torch.stack(xs), torch.stack(xn), torch.stack(y)


# ============================================================================
# 5. Training loop
# ============================================================================
DL_DEFAULT: Dict[str, Any] = dict(
    d_model=128, n_transformer_layers=2, pure_cnn=False, pure_transformer=False,
    dropout=0.2, max_seq_len=20000, vocab_max=None, token_aug_prob=0.0,
    use_num_features=True, use_sequence=True,
    loss="bce", use_pos_weight=False, focal_gamma=2.0,
    asym_gamma_pos=0.0, asym_gamma_neg=4.0, asym_clip=0.05,
    epochs=20, batch_size=32, lr_base=3e-5, lr_max=3e-4,
    pct_start=0.1, weight_decay=1e-4, patience=5,
    tune_thresholds=False, num_workers=0, seed=42,
)


def _train_one_epoch(model, loader, opt, sched, loss_fn) -> float:
    model.train()
    total, n = 0.0, 0
    for x_seq, x_num, y in loader:
        x_seq = x_seq.to(DEVICE); x_num = x_num.to(DEVICE); y = y.to(DEVICE)
        opt.zero_grad()
        logits = model(x_seq, x_num)
        loss = loss_fn(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        total += loss.item() * y.size(0); n += y.size(0)
    return total / max(n, 1)


@torch.no_grad()
def _validate(model, loader, loss_fn):
    # Switch to inference mode without using the dot-shorthand method.
    model.train(False)
    total, n = 0.0, 0
    logits_all, labels_all = [], []
    for x_seq, x_num, y in loader:
        x_seq = x_seq.to(DEVICE); x_num = x_num.to(DEVICE); y = y.to(DEVICE)
        lg = model(x_seq, x_num)
        loss = loss_fn(lg, y)
        total += loss.item() * y.size(0); n += y.size(0)
        logits_all.append(lg.cpu()); labels_all.append(y.cpu())
    return total / max(n, 1), torch.cat(logits_all), torch.cat(labels_all)


def _split_internal(token_ids, labels: np.ndarray,
                    nums: np.ndarray, val_frac: float = 0.2,
                    seed: int = 42):
    """Stratified-free random split; works with ndarray OR list[ndarray]."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(labels))
    n_val = int(len(labels) * val_frac)
    vi, ti = perm[:n_val], perm[n_val:]
    if isinstance(token_ids, np.ndarray):
        tr_ids, va_ids = token_ids[ti], token_ids[vi]
    else:
        # Ragged list[ndarray] — fancy indexing not supported.
        tr_ids = [token_ids[i] for i in ti]
        va_ids = [token_ids[i] for i in vi]
    return (tr_ids, labels[ti], nums[ti],
            va_ids, labels[vi], nums[vi])


def run_dl_experiment(config: Dict[str, Any],
                      tokenizer: BytecodeTokenizer,
                      train_token_ids: np.ndarray,
                      train_labels: np.ndarray,
                      train_nums: np.ndarray,
                      val_token_ids: np.ndarray,
                      val_labels: np.ndarray,
                      val_nums: np.ndarray,
                      runs_dir: Path,
                      wandb_run=None) -> Dict[str, Any]:
    """Train one DL config; cache to ``runs_dir/{name}.json`` if not present."""
    cfg = {**DL_DEFAULT, **config}
    name = cfg["name"]
    cache = Path(runs_dir) / f"{name}.json"
    if cache.exists():
        print(f" SKIP {name} (cached)")
        return json.loads(cache.read_text(encoding="utf-8"))

    print(f"\n{'='*60}\nDL run: {name}\n{'='*60}")
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    gen = torch.Generator().manual_seed(cfg["seed"])

    int_tr_ids, int_tr_y, int_tr_n, int_va_ids, int_va_y, int_va_n = \
        _split_internal(train_token_ids, train_labels, train_nums,
                        val_frac=0.2, seed=cfg["seed"])

    tr_ds = BytecodeDataset(int_tr_y, int_tr_n, cfg["max_seq_len"], int_tr_ids,
                            token_aug_prob=cfg["token_aug_prob"])
    va_ds = BytecodeDataset(int_va_y, int_va_n, cfg["max_seq_len"], int_va_ids)
    kw = dict(batch_size=cfg["batch_size"], num_workers=cfg["num_workers"],
              pin_memory=True, collate_fn=collate, generator=gen)
    tr_loader = DataLoader(tr_ds, shuffle=True, **kw)
    va_loader = DataLoader(va_ds, shuffle=False, **kw)

    n_num = train_nums.shape[1]
    model = MultilabelClassifier(tokenizer.vocab_size, n_num,
                                  train_labels.shape[1], cfg).to(DEVICE)
    n_p = sum(p.numel() for p in model.parameters())
    print(f"  params={n_p/1e6:.2f}M · train={len(tr_ds)} val={len(va_ds)}")

    # pos_weight is computed from the internal training split (int_tr_y),
    # NOT the full train_labels — including the held-out 20% inflates the
    # positive count and biases pos_weight downward by ~20%.
    pos_w = (compute_pos_weight(torch.tensor(int_tr_y), DEVICE)
             if cfg["use_pos_weight"] else None)
    loss_fn = get_loss_fn(cfg, pos_weight=pos_w)

    decay = [p for _, p in model.named_parameters()
             if p.requires_grad and p.dim() >= 2]
    nodec = [p for _, p in model.named_parameters()
             if p.requires_grad and p.dim() < 2]
    opt = AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                 {"params": nodec, "weight_decay": 0.0}], lr=cfg["lr_base"])
    sched = OneCycleLR(opt, max_lr=cfg["lr_max"],
                       steps_per_epoch=len(tr_loader),
                       epochs=cfg["epochs"], pct_start=cfg["pct_start"],
                       anneal_strategy="cos")

    t0 = time.time()
    best_f1, best_epoch = 0.0, -1
    best_state = None
    best_va_logits, best_va_labels = None, None
    history: List[Dict[str, float]] = []

    for ep in range(1, cfg["epochs"] + 1):
        tl = _train_one_epoch(model, tr_loader, opt, sched, loss_fn)
        vl, vlog, vlab = _validate(model, va_loader, loss_fn)
        m = compute_dl_metrics(vlog, vlab)
        f1 = m["f1_macro"]
        history.append({"epoch": ep, "tr_loss": tl, "vl_loss": vl,
                        "f1_macro": f1, "subset_acc": m["subset_acc"]})
        star = ""
        if f1 > best_f1:
            best_f1, best_epoch = f1, ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_va_logits, best_va_labels = vlog.clone(), vlab.clone()
            star = "  "
        if wandb_run is not None:
            # No `step=` — W&B step must be monotonic across the whole run;
            # using per-experiment epoch numbers would collide between
            # experiments.  Letting W&B auto-increment is the clean fix.
            wandb_run.log({
                f"dl/{name}/train_loss": tl,
                f"dl/{name}/val_loss":   vl,
                f"dl/{name}/macro_f1":   f1,
                f"dl/{name}/subset_acc": m["subset_acc"],
                f"dl/{name}/epoch":      ep,
            })
        print(f"  [{ep:2d}/{cfg['epochs']}] tr={tl:.4f} vl={vl:.4f} f1={f1:.4f}{star}")
        if ep - best_epoch >= cfg["patience"]:
            print(f"  early stop after {ep} epochs"); break

    train_time = time.time() - t0
    if best_state is not None:
        model.load_state_dict(best_state)

    ext_ds = BytecodeDataset(val_labels, val_nums, cfg["max_seq_len"],
                             val_token_ids)
    ext_loader = DataLoader(ext_ds, batch_size=cfg["batch_size"],
                            shuffle=False, collate_fn=collate, num_workers=0)
    _, ext_lg, ext_lb = _validate(model, ext_loader, loss_fn)
    ext_default = compute_dl_metrics(ext_lg, ext_lb)
    ext = ext_default
    opt_thr = None
    if cfg["tune_thresholds"] and best_va_logits is not None:
        opt_thr = tune_thresholds(torch.sigmoid(best_va_logits), best_va_labels)
        ext = compute_dl_metrics(ext_lg, ext_lb, thresholds=opt_thr)

    result = {
        "name": name,
        "config": {k: v for k, v in cfg.items() if k != "name"},
        "best_epoch": best_epoch, "best_f1_internal": best_f1,
        "macro_f1_external": ext["f1_macro"],
        "macro_f1_external_default_thr": ext_default["f1_macro"],
        "subset_acc_external": ext["subset_acc"],
        "f1_per_label": ext["f1_per_label"],
        "precision_per_label": ext["precision_per_label"],
        "recall_per_label": ext["recall_per_label"],
        "optimal_thresholds": opt_thr.tolist() if opt_thr is not None else None,
        "train_time_sec": train_time, "total_epochs": len(history),
        "history": history, "model_params_M": n_p / 1e6,
        "family": "DL", "task": "multi-label",
    }
    cache.write_text(json.dumps(result, indent=2, ensure_ascii=False),
                     encoding="utf-8")
    if wandb_run is not None:
        wandb_run.summary.update({
            f"dl/{name}/macro_f1_external": result["macro_f1_external"],
            f"dl/{name}/train_time_min":    result["train_time_sec"] / 60,
            f"dl/{name}/best_epoch":        result["best_epoch"],
            f"dl/{name}/n_params_M":        result["model_params_M"],
        })
    print(f"\n{name}: F1_ext={ext['f1_macro']:.4f} · {train_time/60:.1f} min")
    # Free GPU memory between experiments — without this the heap fragments
    # across 10 sequential runs and later (C2_dmodel_256, C4_pure_transformer)
    # configurations OOM.
    del model, opt, sched, best_state
    if best_va_logits is not None:
        del best_va_logits, best_va_labels
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


# ============================================================================
# 6. Experiment registry
# ============================================================================
DL_EXPERIMENTS: List[Dict[str, Any]] = [
    {"name": "A1_baseline"},
    {"name": "B1_pos_weight",       "use_pos_weight": True},
    {"name": "B2_focal_g2",         "loss": "focal", "focal_gamma": 2.0},
    {"name": "B3_focal_pos_weight", "loss": "focal", "focal_gamma": 2.0,
                                    "use_pos_weight": True},
    {"name": "B4_asymmetric",       "loss": "asymmetric",
                                    "asym_gamma_pos": 0.0, "asym_gamma_neg": 4.0,
                                    "asym_clip": 0.05},
    {"name": "B5_threshold_tuning", "use_pos_weight": True,
                                    "tune_thresholds": True},
    {"name": "C1_transformer_4layers", "use_pos_weight": True,
                                       "n_transformer_layers": 4},
    {"name": "C2_dmodel_256",          "use_pos_weight": True, "d_model": 256},
    {"name": "C3_pure_cnn",            "use_pos_weight": True, "pure_cnn": True},
    {"name": "C4_pure_transformer",    "use_pos_weight": True,
                                       "pure_transformer": True,
                                       # OOM-safe on P100 16GB: full O(L^2)
                                       # attention with L=2048 needs ~1 GB per
                                       # batch element; batch_size=4 keeps
                                       # peak GPU memory ~6 GB.
                                       "max_seq_len": 2048, "batch_size": 4},
]


def run_all_dl(tokenizer: BytecodeTokenizer,
               train_token_ids: np.ndarray, train_labels: np.ndarray,
               train_nums: np.ndarray,
               val_token_ids: np.ndarray, val_labels: np.ndarray,
               val_nums: np.ndarray, runs_dir: Path,
               wandb_run=None, time_budget_sec: float = None,
               reverse_order: bool = False) -> List[Dict[str, Any]]:
    """Run all 10 DL configurations; defensive: failures are isolated.

    Cross-run resume: any config whose ``runs_dir/{name}.json`` already exists
    (restored from the dl-ablation-cache dataset) is returned instantly from
    cache. ``time_budget_sec`` lets a single Kaggle session stop launching NEW
    (uncached) configs once the wall-clock budget is hit, so the run commits its
    partial cache before Kaggle's 12 h hard cap. Re-running with the updated
    cache resumes where it left off; once all 10 are cached the notebook runs
    end-to-end in minutes.
    """
    print(f"\n{'#'*64}\n# DL ablation - {len(DL_EXPERIMENTS)} configs"
          + (f" - budget {time_budget_sec/3600:.1f} h this session" if time_budget_sec else "")
          + f"\n{'#'*64}")
    results: List[Dict[str, Any]] = []
    t_start = time.time()
    _experiments = list(reversed(DL_EXPERIMENTS)) if reverse_order else list(DL_EXPERIMENTS)
    for cfg in _experiments:
        name = cfg["name"]
        cached = (Path(runs_dir) / f"{name}.json").exists()
        if (not cached and time_budget_sec is not None
                and (time.time() - t_start) > time_budget_sec):
            remaining = [c["name"] for c in DL_EXPERIMENTS
                         if not (Path(runs_dir) / (c["name"] + ".json")).exists()]
            print("\n[TIME BUDGET] {:.1f} h reached - stopping before {}. "
                  "Re-run to resume (remaining: {}).".format(
                      time_budget_sec / 3600, name, remaining))
            break
        try:
            r = run_dl_experiment(cfg, tokenizer,
                                  train_token_ids, train_labels, train_nums,
                                  val_token_ids, val_labels, val_nums,
                                  runs_dir, wandb_run=wandb_run)
            results.append(r)
        except Exception as e:
            print(f" {cfg.get('name','?')} failed: {type(e).__name__}: {e}")
            if wandb_run is not None:
                try:
                    wandb_run.summary.update(
                        {f"dl/{cfg['name']}/error": str(e)[:200]})
                except Exception:
                    pass
    done = sorted(p.stem for p in Path(runs_dir).glob("*.json")
                  if p.stem in {c["name"] for c in DL_EXPERIMENTS})
    print(f"\n=== DL ablation: {len(done)}/{len(DL_EXPERIMENTS)} configs now cached "
          f"({len(results)} available this session) ===")
    return results


__all__ = [
    "DEVICE", "MAX_SEQ_LEN_FULL", "BytecodeTokenizer", "build_token_ids",
    "ConvBlock", "AttentionPooling", "BytecodeEncoder", "MultilabelClassifier",
    "compute_pos_weight", "get_loss_fn", "compute_dl_metrics", "tune_thresholds",
    "BytecodeDataset", "collate", "DL_DEFAULT",
    "run_dl_experiment", "DL_EXPERIMENTS", "run_all_dl",
]


In [ ]:
# 6. Time the paper's multi-label XGBoost. Same constructor as 02_code/05_analyze_v2.py, full training split.
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score
import xgboost as xgb

xgb_ctor = lambda: MultiOutputClassifier(
    xgb.XGBClassifier(n_estimators=20 if SMOKE else 400, max_depth=8, n_jobs=-1,
                      random_state=SEED, eval_metric="logloss"), n_jobs=1)
m = xgb_ctor()
t0 = time.perf_counter(); m.fit(X_tr, y_tr.astype("int8")); xgb_fit_s = time.perf_counter() - t0
t0 = time.perf_counter(); P = m.predict(X_va); xgb_pred_s = time.perf_counter() - t0
xgb_val_macro = float(np.mean([f1_score(y_va[:, i], P[:, i]) for i in range(8)]))
print(f"multi-label XGBoost: fit {xgb_fit_s:.1f} s on {os.cpu_count()} CPUs | predict {len(X_va):,} rows in {xgb_pred_s:.2f} s | validation macro-F1 {xgb_val_macro:.4f}")
timing = {"model": "ML2_XGBoost", "n_estimators": 20 if SMOKE else 400, "max_depth": 8,
          "fit_seconds": xgb_fit_s, "predict_seconds_val": xgb_pred_s, "cpus": os.cpu_count(),
          "platform": platform.platform(), "val_macro_f1": xgb_val_macro, "train_rows": int(len(X_tr)), "val_rows": int(len(X_va))}
(OUT_DIR / "xgb_timing.json").write_text(json.dumps(timing, indent=2))
del m, P; gc.collect()


In [ ]:
# 7. C2 on the decoded opcodes. The two arms are encoded and trained one after the other, not
#    together: the ragged token ids of both representations plus the re-attached bytecode would be
#    several gigabytes at once on a session that has about thirteen. The row set is fixed here,
#    before either arm, so both still see exactly the same contracts.
#    The conv stack has fixed strides (4*5*4*2 = 160) and a hard-coded 125-position mask, so
#    max_seq_len stays 20000 for both representations; smoke mode shrinks rows and epochs only.
C2 = next(c for c in DL_EXPERIMENTS if c["name"] == "C2_dmodel_256")
OVERRIDE = {"epochs": 1, "batch_size": 8} if SMOKE else {}
MAX_LEN = 20000

keep_tr = np.array([k in code_by_key for k in keys["train"]])
keep_va = np.array([k in code_by_key for k in keys["val"]])
y_tr_dl, X_tr_dl = y_tr[:n_tr_dl][keep_tr], X_tr[:n_tr_dl][keep_tr]
y_va_dl, X_va_dl = y_va[:n_va_dl][keep_va], X_va[:n_va_dl][keep_va]
recovery.update({"dl_train_rows": int(keep_tr.sum()), "dl_val_rows": int(keep_va.sum())})
print(f"rows for both arms: train {keep_tr.sum():,} | val {keep_va.sum():,}")

def encode(texts, tok):
    ids = [tok.encode_unpadded(s, MAX_LEN) for s in texts]
    lens = np.array([len(x) for x in ids])
    print(f"  encoded {len(ids):,} | length median {int(np.median(lens)):,} | truncated at {MAX_LEN}: {(lens >= MAX_LEN).sum():,}")
    return ids

def decoded_texts(split_keys):
    for k in split_keys:
        if k in code_by_key:
            yield decoded_text(k)[0]

def legacy_texts(name, keep, n_rows):
    """Selects on the row mask, not on code_by_key: that map is freed with the decoded arm."""
    for s, wanted in zip(iter_text(name, n_rows), keep):
        if wanted:
            yield s

def run_c2(tr_ids, va_ids, tok, tag):
    cfg = {**C2, **OVERRIDE, "name": f"C2_dmodel_256_{tag}"}
    t0 = time.time()
    res = run_dl_experiment(cfg, tok, tr_ids, y_tr_dl, X_tr_dl, va_ids, y_va_dl, X_va_dl, RUNS_DIR)
    res["wall_seconds_incl_eval"] = time.time() - t0
    res["representation"] = tag; res["external_split"] = "val_v2"; res["vocab_size"] = tok.vocab_size
    res["train_rows"] = int(len(tr_ids)); res["val_rows"] = int(len(va_ids))
    (OUT_DIR / f"c2_{tag}.json").write_text(json.dumps(res, indent=2))
    print(f"[{tag}] C2 validation macro-F1 {res['macro_f1_external']:.4f} | best epoch {res['best_epoch']} of {res['total_epochs']} | {res['train_time_sec']/60:.1f} min")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return res

# The tokeniser is fitted on the first 20,000 training rows, as in the paper. `fit` only iterates,
# so it is fed a generator: materialising 20,000 legacy strings costs more than a gigabyte.
t0 = time.time()
tok_dec = BytecodeTokenizer().fit(decoded_texts(keys["train"][:20000]))
print(f"[decoded] vocabulary {tok_dec.vocab_size} | fitted in {time.time()-t0:.0f}s")
tr_dec = encode(decoded_texts(keys["train"]), tok_dec)
va_dec = encode(decoded_texts(keys["val"]), tok_dec)
metadata_stripped = sum(decoded_text(k)[1] for k in keys["train"] if k in code_by_key)
recovery["metadata_stripped_train"] = int(metadata_stripped)
del code_by_key; gc.collect()      # the bytecode is not needed once the decoded arm is encoded
res_decoded = run_c2(tr_dec, va_dec, tok_dec, "decoded")
del tr_dec, va_dec; gc.collect()


In [ ]:
# 8. Control: the same C2 on the legacy tokens, same rows, seed and internal split. Only the
#    representation differs. Encoded here rather than in cell 7 so the two never share memory.
if RUN_LEGACY_CONTROL:
    t0 = time.time()
    tok_leg = BytecodeTokenizer().fit(legacy_texts("train_v2.parquet", keep_tr[:20000], 20000))
    print(f"[legacy] vocabulary {tok_leg.vocab_size} | fitted in {time.time()-t0:.0f}s")
    tr_leg = encode(legacy_texts("train_v2.parquet", keep_tr, n_tr_dl), tok_leg)
    va_leg = encode(legacy_texts("val_v2.parquet", keep_va, n_va_dl), tok_leg)
    res_legacy = run_c2(tr_leg, va_leg, tok_leg, "legacy")
    del tr_leg, va_leg; gc.collect()
else:
    res_legacy = None


In [ ]:
# 9. Results
ref = {"legacy_C2_val_full_run_v12": 0.6578, "xgb_val_full_run_v12": 0.7518, "legacy_C2_test_paper": 0.6793}
rows = [("XGBoost, this run (val)", xgb_val_macro, f"fit {xgb_fit_s:.0f} s on {os.cpu_count()} CPUs"),
        ("XGBoost, released run (val)", ref["xgb_val_full_run_v12"], "reference"),
        ("C2 decoded opcodes, this run (val)", res_decoded["macro_f1_external"], f"{res_decoded['train_time_sec']/60:.0f} min, {res_decoded['total_epochs']} epochs")]
if res_legacy:
    rows.append(("C2 legacy tokens, this run (val)", res_legacy["macro_f1_external"], f"{res_legacy['train_time_sec']/60:.0f} min, {res_legacy['total_epochs']} epochs"))
rows.append(("C2 legacy tokens, released run (val)", ref["legacy_C2_val_full_run_v12"], "reference"))
print(f"{'model':42s} {'macro-F1':>9s}   note"); print("-" * 72)
for name, v, note in rows: print(f"{name:42s} {v:9.4f}   {note}")
pick = lambda r: {k: r[k] for k in ("macro_f1_external", "best_epoch", "total_epochs", "train_time_sec", "vocab_size", "f1_per_label", "train_rows", "val_rows")}
summary = {"smoke": SMOKE, "external_split": "val_v2", "test_split_opened": False,
           "bytecode_recovery": recovery, "feature_provenance": provenance, "xgb_timing": timing,
           "c2_decoded": pick(res_decoded), "c2_legacy": pick(res_legacy) if res_legacy else None,
           "reference_points": ref, "dl_pipeline_sha256": "e37b1594fa66d07cd2a3e4335373bbf5a15fff45dc937b337800c787b42d75f1",
           "environment": {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                           "torch": torch.__version__, "xgboost": xgboost.__version__, "python": platform.python_version()}}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print("\nwritten:", sorted(p.name for p in OUT_DIR.glob("*.json")))

# Where each number this notebook contributes appears in the paper. The paper cites numbers only
# through LaTeX macros generated from these files, so this is the whole mapping for this run.
print("\n" + "=" * 72)
print("every number this run contributes to the paper")
print("=" * 72)
gap = abs(100 * (res_decoded["macro_f1_external"] - res_legacy["macro_f1_external"])) if res_legacy else float("nan")
xgap = abs(100 * (xgb_val_macro - res_decoded["macro_f1_external"]))
hit = recovery["train_rows_reattached"] + recovery["val_rows_reattached"]
tot = recovery["train_rows"] + recovery["val_rows"]
for name, value, source in [
    ("vDecodedVal", f"{res_decoded['macro_f1_external']:.4f}", "cell 7, C2 on decoded opcodes, val_v2 macro-F1"),
    ("vLegacyVal", f"{res_legacy['macro_f1_external']:.4f}" if res_legacy else "-", "cell 8, same C2 on legacy tokens"),
    ("vDecXgbVal", f"{xgb_val_macro:.4f}", "cell 6, multi-label XGBoost, val_v2 macro-F1"),
    ("vXgbFitS", f"{xgb_fit_s:.0f}", "cell 6, wall clock of the XGBoost fit, seconds"),
    ("vXgbCpus", f"{os.cpu_count()}", "cell 6, CPU cores of this session"),
    ("vDecodedMin", f"{res_decoded['train_time_sec']/60:.0f}", "cell 7, GPU training minutes"),
    ("vLegacyMin", f"{res_legacy['train_time_sec']/60:.0f}" if res_legacy else "-", "cell 8, GPU training minutes"),
    ("vDecGapAbs", f"{gap:.2f}", "decoded minus legacy, percentage points"),
    ("vDecXgbGapAbs", f"{xgap:.2f}", "XGBoost minus decoded, percentage points"),
    ("vDecTrainRows", f"{res_decoded['train_rows']:,}", "cell 7, contracts both arms trained on"),
    ("vDecValRows", f"{res_decoded['val_rows']:,}", "cell 7, contracts both arms scored on"),
    ("vDecRowsPct", f"{100*hit/tot:.1f}%", "cell 3, released rows whose bytecode was re-attached"),
    ("vDecUpstream", f"{recovery['upstream_rows']:,}", "cell 3, upstream contracts scanned"),
    ("vDecVocab", f"{res_decoded['vocab_size']:,}", "cell 7, decoded-opcode vocabulary"),
    ("vDecLenMed", f"{recovery['sample_decoded_len_median']:,}", "cell 3, median decoded tokens per contract"),
]:
    print(f"  {name:16s} {value:>10s}   {source}")
print("\nfeature provenance (cell 4b): "
      f"{provenance['features_exact']}/{provenance['features_total']} released features are an exact "
      f"affine image of the extractor's output on the re-attached bytecode")
print("reference points NOT recomputed here:", ref)
